<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-7025/blob/main/notebooks/tuesday-your-turn/week-05-lecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 5: Data Visualization & EDA 📊

## Today's Mission:
- Choose the right visualization library for the job
- Build charts with Pandas, Seaborn, and Matplotlib using the Complete Journey data
- Apply a systematic EDA framework — question → structure → distributions → segmentation → story

**Follow along with the slides and complete the challenges below!**

## Getting Started: Load the Data

We'll use the Complete Journey dataset throughout today. Complete Journey Docs: [bit.ly/completejourney_py](https://cunningjames.github.io/completejourney_py/)

In [ ]:
# You may need to install the package first
# !pip install completejourney-py

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from completejourney_py import get_data

cj = get_data()
transactions  = cj['transactions']
products      = cj['products']
demographics  = cj['demographics']

print("Datasets loaded:", list(cj.keys()))

In [ ]:
# Pre-build the DataFrames we'll use throughout

# Basket-level spend
basket_spend = (
    transactions
    .groupby(['household_id', 'basket_id'], as_index=False)
    ['sales_value'].sum()
    .rename(columns={'sales_value': 'basket_spend'})
)

# Basket spend joined to demographics
basket_demo = basket_spend.merge(demographics, on='household_id', how='inner')

# Weekly sales
weekly_sales = (
    transactions
    .set_index('transaction_timestamp')['sales_value']
    .resample('W').sum()
    .reset_index()
    .rename(columns={'transaction_timestamp': 'week', 'sales_value': 'total_sales'})
)

# Top 10 departments by revenue
category_totals = (
    transactions
    .merge(products[['product_id', 'department']], on='product_id')
    .groupby('department', as_index=False)['sales_value'].sum()
    .sort_values('sales_value')
    .tail(10)
)

income_order = [
    'Under 15K','15-24K','25-34K','35-49K',
    '50-74K','75-99K','100-124K','125-149K',
    '150-174K','175-199K','200-249K','250K+'
]

print("Ready! basket_spend, basket_demo, weekly_sales, category_totals all loaded.")

---

# Part 1: Pandas Visualization 🐼

Pandas `.plot()` is your fastest path from a DataFrame to a chart — one line, no imports, good enough for EDA.

## The Mental Model

**One variable** → call `.plot()` on a **Series**, specify `kind=`:
- `kind='hist'` — distribution
- `kind='box'` — spread and outliers
- `kind='line'` — trend (index is x-axis)

**Two variables** → call `.plot()` on a **DataFrame**, add `x=` and `y=`:
- `kind='scatter'` — relationship between two numeric columns
- `kind='bar'` / `kind='barh'` — category comparison
- `kind='line'` — trend over time

## Example: Distribution

In [ ]:
basket_spend['basket_spend'].plot(
    kind='hist', bins=40, figsize=(10, 3.5),
    title='Distribution of basket spend', xlabel='Basket spend ($)'
)
plt.tight_layout()
plt.show()

## Example: Category Comparison

In [ ]:
category_totals.plot(
    kind='barh', x='department', y='sales_value',
    figsize=(10, 3.5), legend=False,
    title='Total revenue by department (top 10)', xlabel='Total sales ($)'
)
plt.tight_layout()
plt.show()

## Example: Trend Over Time

In [ ]:
weekly_sales.plot(
    kind='line', x='week', y='total_sales',
    figsize=(10, 3.2), legend=False,
    title='Weekly total sales', ylabel='Total sales ($)'
)
plt.tight_layout()
plt.show()

## 🧑‍💻 Exercise 1 — Try It

Using the DataFrames already loaded above:

### Part A — One variable

Plot the distribution of `basket_spend`. Choose `kind='hist'` or `kind='box'` — whichever you think better shows the shape of the data.

In [ ]:
# Part A: distribution of basket spend
basket_spend['basket_spend'].plot(
    kind=_______,        # 'hist' or 'box'
    figsize=(10, 3.5),
    title=_______        # give it a descriptive title
)
plt.tight_layout()
plt.show()

### Part B — Two variables

`basket_demo` has both `basket_spend` and `marital_status`. Aggregate to get average basket spend per marital status group, then plot a bar chart comparing the groups.

In [ ]:
# Part B: average basket spend by marital status
avg_by_marital = (
    basket_demo
    .groupby('_______', as_index=False)    # group by marital_status
    ['basket_spend']._______()             # compute the mean
)

avg_by_marital.plot(
    kind='_______',                        # 'bar' or 'barh'
    x='marital_status',
    y='basket_spend',
    figsize=(8, 3.5),
    legend=False,
    title='Average basket spend by marital status'
)
plt.tight_layout()
plt.show()

**Discussion:** What do you see? Is the difference meaningful, or does it look like noise?

---

# Part 2: Seaborn Visualization 📈

Seaborn is built on Matplotlib and gives you statistical chart types — grouping, ordering, and distributions — with much less code.

## The Mental Model

Every Seaborn function follows the same pattern:

```python
sns.function(data=df, x='col', y='col', hue='group', order=[...])
```

| Argument | Role |
|----------|------|
| `data=` | the DataFrame — always required |
| `x=` | horizontal axis |
| `y=` | vertical axis (two-variable plots) |
| `hue=` | color by group — adds a third dimension |
| `order=` | control category order — critical for income, size, etc. |

## Example: Distribution with KDE — `histplot`

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
sns.histplot(
    data=basket_demo,
    x='basket_spend',
    bins=50,
    kde=True,
    ax=ax
)
ax.set_xlim(0, 100)
ax.set_xlabel('Basket spend ($)')
ax.set_title('Distribution of basket spend — with density curve')
plt.tight_layout()
plt.show()

## Example: Group Comparison — `boxplot`

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    data=basket_demo,
    x='income',
    y='basket_spend',
    order=income_order,
    ax=ax,
    showfliers=False
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Income bracket')
ax.set_ylabel('Basket spend ($)')
ax.set_title('Basket spend by income bracket')
plt.tight_layout()
plt.show()

## Example: Matrix Pattern — `heatmap`

In [ ]:
# Build day × hour heatmap data
transactions['hour'] = transactions['transaction_timestamp'].dt.hour
transactions['day_of_week'] = transactions['transaction_timestamp'].dt.day_name()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = (
    transactions
    .groupby(['day_of_week', 'hour'])
    .size()
    .reset_index(name='trip_count')
    .pivot(index='day_of_week', columns='hour', values='trip_count')
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(
    heatmap_data.iloc[:, 6:22],
    cmap='YlOrRd',
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Trip count'}
)
ax.set_xlabel('Hour of day')
ax.set_ylabel('')
ax.set_title('Shopping trips by day of week and hour')
plt.tight_layout()
plt.show()

## 🔍 Exercise 2 — Seaborn Gallery Hunt

Browse the **[Seaborn example gallery](https://seaborn.pydata.org/examples/)** and find one plot that you think could reveal something interesting about the Complete Journey data.

Record your findings below:

**Plot type I chose:** *Write here*

**CJ variables I would use:** *e.g., x='income', y='basket_spend', hue='marital_status'*

**Insight I hope it reveals:** *Write here*

---

# Part 3: Matplotlib Visualization 🎨

Matplotlib gives you complete control — every label, tick, annotation, and spine is yours to modify. Use it when the chart needs to stand on its own in a report or presentation.

## The Figure / Axes Hierarchy

```python
fig, ax = plt.subplots(figsize=(10, 4))
# fig = the overall canvas
# ax  = the plot area — axes, ticks, lines, labels all live here
```

Get handles for both, and you can modify anything.

## Step 0: Quick Pandas Starting Point

In [ ]:
# Fast and rough — good enough to see the shape
weekly_sales.plot(kind='line', x='week', y='total_sales', figsize=(10, 3), legend=False)
plt.show()

## Step 1: Move to the Figure / Axes API

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(weekly_sales['week'], weekly_sales['total_sales'], linewidth=2)
ax.set_title('Weekly total sales')
ax.set_xlabel('Week')
ax.set_ylabel('Total sales ($)')
plt.tight_layout()
plt.show()

## Step 2: Executive-Ready Formatting

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(weekly_sales['week'], weekly_sales['total_sales'], linewidth=2)
ax.set_title('Weekly total sales — Regork grocery chain', pad=10)
ax.set_xlabel('Week')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))  # currency format
ax.grid(True, alpha=0.3)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)  # clean look
plt.tight_layout()
plt.show()

## Step 3: Highlight the Insight — Annotations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(weekly_sales['week'], weekly_sales['total_sales'], linewidth=2)
ax.set_title('Weekly total sales — note holiday spike in late December', pad=10)
ax.set_xlabel('Week')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.grid(True, alpha=0.3)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Find and annotate the peak week
peak_idx  = weekly_sales['total_sales'].idxmax()
peak_week = weekly_sales.loc[peak_idx, 'week']
peak_val  = weekly_sales.loc[peak_idx, 'total_sales']

ax.annotate(
    f'Holiday spike\n${peak_val:,.0f}',
    xy=(peak_week, peak_val),
    xytext=(peak_week - pd.Timedelta(weeks=8), peak_val * 0.97),
    arrowprops=dict(arrowstyle='->', lw=1.2),
    fontsize=9
)
plt.tight_layout()
plt.show()

## Matplotlib Is the Foundation

Because Seaborn returns a Matplotlib `ax`, you can mix them — Seaborn draws the chart, Matplotlib polishes it:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Seaborn draws the plot
sns.boxplot(
    data=basket_demo, x='income', y='basket_spend',
    order=income_order, ax=ax, showfliers=False, color='steelblue'
)

# Matplotlib polishes it
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Income bracket')
ax.set_ylabel('Basket spend ($)')
ax.set_title('Basket spend by income — Seaborn + Matplotlib finishing touches')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

## 🔍 Exercise 3 — Matplotlib Gallery Hunt

Browse the **[Matplotlib example gallery](https://matplotlib.org/stable/gallery/index.html)** and find one plot that you think could reveal something interesting about the Complete Journey data.

Record your findings below:

**Plot type I chose:** *Write here*

**CJ variables I would use:** *Write here*

**Insight I hope it reveals:** *Write here*

---

# Part 4: Bokeh — Interactive Visualization 🌐

Bokeh renders to HTML and JavaScript — charts live in the browser, not just the notebook. Users can zoom, pan, and hover without writing new code.

In [ ]:
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.io import output_notebook

output_notebook()

In [ ]:
source = ColumnDataSource(weekly_sales)

p = figure(
    title='Total Weekly Sales — hover, zoom, and pan to explore',
    x_axis_type='datetime',
    width=750, height=350,
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p.line('week', 'total_sales', source=source, line_width=2, color='steelblue')

hover = HoverTool(tooltips=[
    ('Week',  '@week{%F}'),
    ('Sales', '@total_sales{$0,0}')
], formatters={'@week': 'datetime'})
p.add_tools(hover)

p.xaxis.axis_label = 'Week'
p.yaxis.axis_label = 'Total Sales ($)'

show(p)

**Try it:** Hover over the spike near the end of the year. Use box zoom to focus on a specific month. What do you notice that you couldn't see in the static Matplotlib version?

---

# Part 5: How to Think About EDA 🔍

EDA is not a checklist of charts to run. It's a **sequence of questions** — each answer raising the next.

| Step | Question to ask |
|------|-----------------|
| **1. Question** | What do I actually want to know? Is it specific enough to know when I've answered it? |
| **2. Structure** | What shape is the data? What do I need to join? Where are the missings? |
| **3. Distributions** | What does the outcome variable look like overall? Any outliers? |
| **4. Segmentation** | Does the pattern differ across groups? Where does the interesting variation live? |
| **5. Story** | What changed? What surprised me? What should the decision-maker do next? |

## Mini EDA: Basket Spend by Income

**Sharp question:** *Do higher-income households spend more per basket — and is the relationship linear or does it flatten?*

In [ ]:
# Step 2: Structure — how many baskets? how many matched households?
print(f"Total baskets:                    {basket_spend['basket_id'].nunique():,}")
print(f"Baskets with demographics joined: {basket_demo['basket_id'].nunique():,}")
print(f"Households with demographics:     {basket_demo['household_id'].nunique():,}")
print(f"Median basket spend:              ${basket_demo['basket_spend'].median():.2f}")

In [ ]:
# Step 3: Distribution — what does basket spend look like overall?
fig, ax = plt.subplots(figsize=(10, 3))
sns.histplot(data=basket_demo, x='basket_spend', bins=50, kde=True, ax=ax)
ax.set_xlim(0, 80)
ax.set_xlabel('Basket spend ($)')
ax.set_title('Most baskets are under $30 — strong right skew')
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: Segmentation — how does spend vary by income bracket?
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(
    data=basket_demo, x='income', y='basket_spend',
    order=income_order, showfliers=False, ax=ax
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Income bracket')
ax.set_ylabel('Basket spend ($)')
ax.set_title('Basket spend rises with income — but flattens above $75K')
plt.tight_layout()
plt.show()

## 🤔 Exercise 4 — What's the Next Question?

Based on the chart above, a teammate reports:

> *"Households in the top two income brackets spend 38% more per basket on average than households in the bottom two brackets."*

Name **3 follow-up questions** you would investigate next and explain what each one would reveal.

**Question 1:** *Write here — what would it reveal?*

**Question 2:** *Write here — what would it reveal?*

**Question 3:** *Write here — what would it reveal?*

---

# 🧾 Quick Reference

## Tool Selection

| Purpose | Tool | When to reach for it |
|---------|------|---------------------|
| Quick EDA | Pandas `.plot()` | Spot-check a distribution or trend in one line |
| Statistical comparison | Seaborn | Compare groups, show distributions with shape |
| Polished reporting | Matplotlib | Final figures for reports and presentations |
| Interactive exploration | Bokeh | Stakeholder tools, hover + zoom, shareable HTML |

## Pandas `.plot()` Cheat Sheet

| Task | Code |
|------|------|
| Distribution | `series.plot(kind='hist', bins=40)` |
| Box plot | `series.plot(kind='box')` |
| Bar chart | `df.plot(kind='bar', x='cat', y='val')` |
| Horizontal bar | `df.plot(kind='barh', x='cat', y='val')` |
| Line chart | `df.plot(kind='line', x='date', y='val')` |
| Scatter | `df.plot(kind='scatter', x='col_a', y='col_b')` |

## Seaborn Cheat Sheet

| Task | Code |
|------|------|
| Distribution + KDE | `sns.histplot(data=df, x='col', kde=True)` |
| Group spread | `sns.boxplot(data=df, x='group', y='col', order=[...])` |
| Relationship | `sns.scatterplot(data=df, x='col_a', y='col_b', hue='group')` |
| Matrix | `sns.heatmap(pivot_df, cmap='YlOrRd')` |
| Group means | `sns.barplot(data=df, x='group', y='col', order=[...])` |

## Matplotlib Polish Cheat Sheet

| Task | Code |
|------|------|
| Currency y-axis | `ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))` |
| Rotate tick labels | `ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')` |
| Remove top/right spines | `for s in ['top','right']: ax.spines[s].set_visible(False)` |
| Add annotation | `ax.annotate('text', xy=(x, y), xytext=(xt, yt), arrowprops=dict(...))` |
| Grid | `ax.grid(True, alpha=0.3)` |

**You're ready for Thursday's lab — project ideation! 🚀**